In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("klue/bert-base")

text = "나는 오늘 콜라와 피자를 함께 마셨다."

inputs = tokenizer(text, return_tensors="pt")

print(inputs)

{'input_ids': tensor([[    2,   717,  2259,  3822,  8525,  2522,  8395,  2138,  3655, 10692,
          2062,    18,     3]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [3]:
tokenizer.tokenize(text)

['나', '##는', '오늘', '콜라', '##와', '피자', '##를', '함께', '마셨', '##다', '.']

In [5]:
tokenizer.convert_ids_to_tokens(inputs["input_ids"])

['[CLS]',
 '나',
 '##는',
 '오늘',
 '콜라',
 '##와',
 '피자',
 '##를',
 '함께',
 '마셨',
 '##다',
 '.',
 '[SEP]']

In [8]:
from transformers import AutoModel

model = AutoModel.from_pretrained("klue/bert-base")

model.safetensors: reconstructing file:   0%|          |  0.00B /  445MB            

model.safetensors: downloading bytes:           |  0.00B            

C:\LANG_CHAIN_2026\2026-05-19_KDT_lang_chain\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hurwa\.cache\huggingface\hub\models--klue--bert-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  message += (


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: klue/bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
inputs

{'input_ids': tensor([[    2,   717,  2259,  3822,  8525,  2522,  8395,  2138,  3655, 10692,
          2062,    18,     3]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [14]:
outputs = model(**inputs)

In [16]:
# 문장 1개, 토큰 13개, 벡터 차원 768
# klue bert base라면 hidden size가 768이여서 768차원 벡터가 만들어짐
outputs.last_hidden_state.shape

torch.Size([1, 13, 768])

In [19]:
# 문장을 대표하는 하나의 벡터로 압축시킴
outputs.pooler_output.shape

torch.Size([1, 768])

In [30]:
texts = [
    "도커를 잘 사용하기 위해선 커널의 내부 동작을 이해해야합니다.",
    "요즘 AI 붐이 일어나면서 잘 해야하는 요소가 하나 더 늘었습니다.",
    "집 내부는 생각보다 깨끗했습니다."
]

inputs = tokenizer(
    texts,
    padding=True,
    return_tensors="pt"
)

inputs["input_ids"].shape, inputs["token_type_ids"].shape, inputs["attention_mask"].shape

(torch.Size([3, 21]), torch.Size([3, 21]), torch.Size([3, 21]))

In [37]:
# tokenizer.convert_ids_to_tokens(inputs["input_ids"][2])
inputs["input_ids"] # 4492는 "내부"라는 의미를 가지게 됨

tensor([[    2,   848,  2506,  2138,  1521,  3704, 31302,  9647,  1710,  2781,
          2079,  4492,  7331,  2069,  3923,  8084, 11800,    18,     3,     0,
             0],
        [    2,  4442,  7212,  1178,  2052,  4652, 31369,  1521,  3645,  2205,
          2259,  4787,  2116,  3657,   831,   794,  2359,  2219,  3606,    18,
             3],
        [    2,  1589,  4492,  2259,  3628,  2178,  2062,  5949,  2371,  2219,
          3606,    18,     3,     0,     0,     0,     0,     0,     0,     0,
             0]])

In [38]:
# sequence token ids -> vectors
outputs = model(**inputs)

print(outputs.last_hidden_state.shape)

torch.Size([3, 21, 768])


In [39]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="doya/klue-sentiment-nsmc"
)

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

C:\LANG_CHAIN_2026\2026-05-19_KDT_lang_chain\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hurwa\.cache\huggingface\hub\models--doya--klue-sentiment-nsmc. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  message += (


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  443MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  443MB            

model.safetensors: downloading bytes:           |  0.00B            

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/752k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [40]:
print(classifier("이 영화 너무 재미있었다."))
print(classifier("시간이 너무 아까운 영화였다"))

[{'label': 'LABEL_1', 'score': 0.9925186634063721}]
[{'label': 'LABEL_0', 'score': 0.997893750667572}]


In [48]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "doya/klue-sentiment-nsmc"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

text = "이 영화 진짜 재미있었다."

inputs = tokenizer(
    text,
    return_tensors="pt"
)

print(inputs)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'input_ids': tensor([[   2, 1504, 3771, 4229, 6001, 2359, 2062,   18,    3]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [49]:
type(inputs["input_ids"])

torch.Tensor

In [51]:
tokens = tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0]
)

print(tokens)

['[CLS]', '이', '영화', '진짜', '재미있', '##었', '##다', '.', '[SEP]']


In [59]:
with torch.no_grad():
    outputs = model(**inputs)

print(outputs.logits)
neg_score, pos_score = torch.softmax(outputs.logits[0], dim=0).tolist()

# print(f"""
# neg_score: {neg_score:.4f}
# pos_score: {pos_score:.4f}
# """)

answer = ("pos_score", pos_score) if pos_score > neg_score else ("neg_score", neg_score)

print(answer[0], answer[1])

tensor([[-2.1704,  2.2601]])
pos_score 0.9882320761680603


In [60]:
print(model.config.id2label)

{0: 'LABEL_0', 1: 'LABEL_1'}


In [61]:
model.config

BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForSequenceClassification"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "problem_type": "single_label_classification",
  "tie_word_embeddings": true,
  "transformers_version": "5.17.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 32000
}